# Generación de interbalos de confianza

In [3]:
import pandas as pd 
import numpy as np 
import warnings
warnings.filterwarnings("ignore")

# Nota: Para este ejemplo asumiremos que los datos de entrenamiento son iguales a los datos de prueba 

In [6]:
Y = np.array([[3], [1], [8], [3], [5]])
Y

array([[3],
       [1],
       [8],
       [3],
       [5]])

In [8]:
X = np.array([[1,3], [1,1], [1,5], [1,2], [1,4]])
X

array([[1, 3],
       [1, 1],
       [1, 5],
       [1, 2],
       [1, 4]])

In [10]:
XT_X = np.matmul(np.matrix.transpose(X), X)
XT_X


array([[ 5, 15],
       [15, 55]])

In [12]:
XT_X_inv = np.linalg.inv(XT_X)
XT_X_inv

array([[ 1.1, -0.3],
       [-0.3,  0.1]])

In [14]:
XT_Y = np.matmul(np.matrix.transpose(X), Y)
XT_Y

array([[20],
       [76]])

In [16]:
betas = np.matmul(XT_X_inv, XT_Y)
betas

array([[-0.8],
       [ 1.6]])

In [18]:
# Calculo de Y's pronosticadas
Y_pred = np.matmul(X, betas)
Y_pred

array([[4. ],
       [0.8],
       [7.2],
       [2.4],
       [5.6]])

In [20]:
# Calculo de residuales
Resid = Y - Y_pred
Resid.round(2)

array([[-1. ],
       [ 0.2],
       [ 0.8],
       [ 0.6],
       [-0.6]])

In [24]:
# Calculo de la suma de residuales al cuadrado
RSS = np.matmul(np.matrix.transpose(Resid), Resid)
RSS

array([[2.4]])

In [26]:
# Calculo de la suma total de cuadrados
TSS = np.matmul(np.matrix.transpose(Y), Y) - len(Y)*(Y.mean()**2)
TSS

array([[28.]])

In [28]:
# Calculo de coeficiente de determinación 
R_cuad = float(1- RSS/TSS)
R_cuad

0.9142857142857144

In [30]:
# Calculo de la varianza del error de regresión
s_cuad = RSS / (len(Y) - X.shape[1])
s_cuad

array([[0.8]])

In [32]:
# Desviación estandar del error de regresión
import math
s = math.sqrt(s_cuad)
s

0.8944271909999157

In [36]:
# Obtener el valor critico de la t de Student de tablas
import scipy.stats

# Grados de libertad: n - (k+1)
grados_libertad = len(Y) - X.shape[1]
Confianza = 0.95
q = (1-Confianza) / 2
# La t_critica se obtendrá de un nivel de confianza del 95% (Alfa = 5%)
t_critico = abs(scipy.stats.t.ppf(q, df = grados_libertad))
t_critico

3.1824463052837078

In [42]:
# Vector de valores particulares de X
f = np.array([[1], [7]])
f

array([[1],
       [7]])

In [46]:
Margen_error = t_critico * (s * (float(np.matmul(np.matmul(np.matrix.transpose(f), XT_X_inv), f)) ** 0.5))
Margen_error

3.818935566340449

In [50]:
Pron_puntual = float(np.matmul(np.matrix.transpose(f), betas))
Pron_puntual

10.400000000000006

In [52]:
# Limites del intervalo de confianza
Lim_inferior = Pron_puntual - Margen_error
Lim_superior = Pron_puntual + Margen_error
print("Intervalo de confienza: (", Lim_inferior, ",", Lim_superior, ")")


Intervalo de confienza: ( 6.581064433659557 , 14.218935566340456 )


# Validación de los supuestos de la regresión 

In [55]:
import scipy
Resid

array([[-1. ],
       [ 0.2],
       [ 0.8],
       [ 0.6],
       [-0.6]])

In [57]:
# Calculo de la asimetría en los residuales
skew = float(scipy.stats.skew(Resid, bias = True))
skew

-0.2886751345948135

In [65]:
# Calculo de la curtosis de residuales
kurtosis = float(scipy.stats.kurtosis(Resid, fisher = False))
kurtosis

1.4499999999999993

In [67]:
Jarque_Bera = (len(Y)/6) * (skew ** 2 + (kurtosis - 3) ** 2 / 4)
Jarque_Bera

0.5699652777777785

In [69]:
Nivel_confianza = 0.95
scipy.stats.chi2.ppf(Nivel_confianza, df = 2)

5.991464547107979

Conclusión: Dado que JB no es mayor al nivel critico, no podemos rechazar la hipótesis de existencia de la normalidad en los residuales

# Supuesto 2: Inexistencia de autocorrelación entre residuales

In [73]:
from statsmodels.formula.api import ols

In [75]:
y_df = pd.DataFrame(Y)
x_df = pd.DataFrame(X[:,1:2])
x_df

,0
0,3
1,1
2,5
3,2
4,4


In [77]:
df = pd.concat([y_df, x_df.reindex(y_df.index)], axis = 1)
df.columns = ["Y", "X1"]
df

,Y,X1
0,3,3
1,1,1
2,8,5
3,3,2
4,5,4


In [81]:
# Ajuste de regresión lineal multiple
model = ols("Y ~ X1", data = df).fit()

from statsmodels.stats.stattools import durbin_watson

# Prueba de Durbin-Watson
durbin_watson(model.resid)

1.3666666666666656

Conclusión: Dado que DW no es aprox. igual a 2, podemos pensar que existe autocorelación entre los residuales

# Supuesto 3: Homocedasticidad (Igualdad de varianzas)

In [85]:
ResidCuad = Resid **2
ResidCuad = pd.DataFrame(ResidCuad)
ResidCuad

,0
0,1.00
1,0.04
2,0.64
3,0.36
4,0.36


In [87]:
X1 = df.iloc[:,1]
X1_df = pd.DataFrame(X1)

X1Cuad = X1 ** 2
X1Cuad_df = pd.DataFrame(X1Cuad)

In [91]:
df_Aux = pd.concat([ResidCuad, X1_df.reindex(y_df.index), X1Cuad_df.reindex(y_df.index)], axis = 1)
df_Aux.columns = ["Residual", "X1", "X1Cuad"]
df_Aux

,Residual,X1,X1Cuad
0,1.00,3,9
1,0.04,1,1
2,0.64,5,25
3,0.36,2,4
4,0.36,4,16


In [93]:
# Ajuste de regresión lineal multiple
modelAux = ols("Residual ~ X1 + X1Cuad", data = df_Aux).fit()
RSqAux = modelAux.rsquared
RSqAux

0.5326278659612038

In [95]:
Estadistico = len(Y) * RSqAux
Estadistico

2.663139329806019

In [97]:
Nivel_confianza = 0.95
scipy.stats.chi2.ppf(Nivel_confianza, df = 2)

5.991464547107979

Conclusión: Al no superar el valor critico nuestro estadistico de prueba, no hay evidencia de Heterocedasticidad (desigualdad de varianza de los residuales) 

In [100]:
# Alternativa para la prueba de White

from statsmodels.stats.diagnostic import het_white

white_test = het_white(model.resid, model.model.exog)
print("Estadistico de prueba:", white_test[0])
print("Valor p:", white_test[1])

Estadistico de prueba: 2.66313932980599
Valor p: 0.26406244627058784


Conclusión: Aun nivel Alfa = 5%, como tenemos un valor p superior a Alfa, no podemos rechazar la hipótesis de Hocedasticidad(lo cual implica que no existe evidencia de Heterecedasticidad)

# Supuesto 4: Inexistencia de Multicolinealidad[]

En este caso no aplica realizarla ya que solo tenemos una variable regresora (X1), En modelos con más variables independientes sí habría que realizarla 